# `helipad-extract`

[HeliPaD](https://github.com/DiGS-Corpora/HeliPaD) provides a fully annotated text of the Old Low German _Heliand_, but it does so in the CorpusSearch format, for which to my knowledge there are no general-purpose Python wrappers (but see [NLTK's YCOECorpusReader module](https://github.com/langeslag/ehtc/blob/main/demo/ycoe.ipynb)). This notebook extracts the form of each token along with its POS, lemma, and halfline ID and outputs them to a JSON file `heliand-c.json` as well as a plaintext file `heliand-c.txt`.

In [1]:
import re,json
from pathlib import Path
from git import Repo

Let's first set some variables for the presentation of the plaintext file below:

In [2]:
print_line_numbers = True
caesura_span = '    '
#caesura_span = '\t'

Now we clone the HeliPaD repository and load the individual lines of its sole PSD file into a list:

In [3]:
# Loop in the HTTPS clone point here:
remote = 'https://github.com/DiGS-Corpora/HeliPaD.git'
# Desired target folder name:
local = 'HeliPaD'
# Only clone if the target folder doesn't already exist:
if not(Path(local).is_dir()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [4]:
with open('HeliPaD/heliand.psd') as infile:
    psd = infile.read().splitlines()

We will ignore the syntactical hierarchy for present purposes and extract only word form, POS tag, and lemma, whose encoding may be expressed by the following regular expression:

In [5]:

pattern = re.compile(r"\(([A-Z0-9^+=*$-]*)\s(\w*)-([^)]*)\)")

The final two pieces of information we may want to store are line number and halfline, so we can reconstruct verse lines downstream.

In [6]:
line_boundary = re.compile(r"\(CODE <R_(\d*)")
caesura = re.compile(r"\(CODE <C>")

We'll write our data to disk once we're done, and we'll build in a file check so we can load the JSON back from disk if the file is already there:

In [7]:
json_file = 'heliand-c.json'
if Path(json_file).is_file():
    with open(json_file) as json_data:
        tokens = json.load(json_data)
else:
    tokens = []
    line_num = 1
    halfline = 'a'
    for line in psd:
        token = dict()
        token['verse'] = str(line_num) + halfline
        result = pattern.search(line)
        newline = line_boundary.search(line)
        off_verse = caesura.search(line)
        if result:
            token['form'] = result.group(2)
            token['lemma'] = result.group(3)
            token['pos'] = result.group(1)
            tokens.append(token)
        elif off_verse:
            halfline = 'b'
        elif newline:
            line_num = int(newline.group(1))
            halfline = 'a'
    with open(json_file, 'w', encoding='utf-8') as outfile:
        json.dump(tokens, outfile, ensure_ascii=False, indent=4)

In [8]:
tokens[205]

{'verse': '30a', 'form': 'mildean', 'lemma': 'mildi', 'pos': 'ADJ^A^SG'}

Now for the plaintext reconstruction:

In [9]:
plaintext_file = 'heliand-c.txt'
if Path(plaintext_file).is_file():
    with open(plaintext_file) as f:
        verse_lines = f.read().splitlines()
else:
    verse_lines = []
    for number in range(1, int(tokens[-1]['verse'].rstrip('ab'))):
        hits_a = [i['form'] for i in tokens if i['verse'] == str(number) + 'a']
        hits_b = [i['form'] for i in tokens if i['verse'] == str(number) + 'b']
        if print_line_numbers == True:
            reconstructed_line = str("{:04d}".format(number)) + ' ' + ' '.join(hits_a) + caesura_span + ' '.join(hits_b)
        else:
            reconstructed_line = ' '.join(hits_a) + caesura_span + ' '.join(hits_b)
        verse_lines.append(reconstructed_line)
    with open(plaintext_file, 'w') as outfile:
        outfile.write('\n'.join(verse_lines))

In [10]:
verse_lines[0]

'0001 Manega uuaron    the sia iro mod gespon'

The edition we have just reconstructed is [Sievers 1878](https://archive.org/details/heliandherausgvonsieve), representing the C text ([London, British Library, MS Cotton Caligula A. vii](https://searcharchives.bl.uk/catalog/041-001102326)). C lacks the last fifteen lines found in the other complete witness, M ([Munich, Bayerische Staatsbibliothek, Cgm 25](https://www.digitale-sammlungen.de/de/view/bsb00026305)), and is otherwise especially notable for its many omissions of small words. For an edition of M, with C's omissions italicized and registered in the apparatus, see Behaghel's _Heliand und Genesis_, either the [current edition with Taeger](https://www.degruyterbrill.com/document/doi/10.1515/9783110963663/) or [an earlier, now public domain edition](https://archive.org/details/heliandundgenesi00beha/) for greater ease of access.